# 08 -- Utility files

Environment setup/imports, the `optimize_dtypes` memory-reduction helper, the `set_seed` reproducibility helper, and the appendix's repo-root path resolver.

In [ ]:
import os
import sys
from pathlib import Path


def _find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "backend" / "app").is_dir() and (candidate / "data").is_dir():
            return candidate
        alt = candidate / "Olist_Marketplace_Platform"
        if (alt / "backend" / "app").is_dir() and (alt / "data").is_dir():
            return alt
    raise RuntimeError("Could not locate the project root above this notebook.")


PROJECT_ROOT = _find_project_root(Path.cwd())
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "backend"))
print("Project root:", PROJECT_ROOT)


In [ ]:
# Standalone setup: a small real `datasets` dict (using the lightweight demo
# sample under data/sample/, not the full raw CSVs) so optimize_dtypes below has
# real dataframes to demonstrate on.
import pandas as pd

datasets = {
    "customers": pd.read_csv("data/sample/olist_customers_dataset.csv"),
    "items": pd.read_csv("data/sample/olist_order_items_dataset.csv"),
    "payments": pd.read_csv("data/sample/olist_order_payments_dataset.csv"),
    "reviews": pd.read_csv("data/sample/olist_order_reviews_dataset.csv"),
    "orders": pd.read_csv("data/sample/olist_orders_dataset.csv"),
    "products": pd.read_csv("data/sample/olist_products_dataset.csv"),
    "sellers": pd.read_csv("data/sample/olist_sellers_dataset.csv"),
}


## 1. Environment Setup & Library Imports
Initialize the project environment and import core libraries

In [1]:
import os
import glob
import numpy as np
import pandas as pd

import plotly.express as px
import plotly.io as pio
import plotly.graph_objects as go

# Robust renderer choice: works without internet / special IDE plugins.
pio.renderers.default = "notebook"
pio.templates.default = "plotly_dark"

import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

print("Libraries loaded successfully.")


Libraries loaded successfully.


### 3.5 Memory Optimization

The raw tables carry a lot of `int64`/`float64` columns that don't need that much
precision (e.g. IDs as strings, small counts). Downcasting reduces memory footprint
before the heavy merge step — useful since `items`/`payments`/`reviews` can be large.

In [18]:
def optimize_dtypes(d: pd.DataFrame):
    before = d.memory_usage(deep=True).sum() / 1024**2

    for col in d.select_dtypes(include="float64").columns:
        d[col] = pd.to_numeric(d[col], downcast="float")

    for col in d.select_dtypes(include="int64").columns:
        d[col] = pd.to_numeric(d[col], downcast="integer")

    after = d.memory_usage(deep=True).sum() / 1024**2
    return d, before, after


total_before = 0
total_after = 0

for name in ["customers", "items", "payments", "reviews",
             "orders", "products", "sellers"]:

    datasets[name], before, after = optimize_dtypes(datasets[name])

    total_before += before
    total_after += after

    print(f"{name:<12} {before:.2f} MB -> {after:.2f} MB")

print(
    f"\nTOTAL: {total_before:.2f} MB -> {total_after:.2f} MB "
    f"({(1-total_after/total_before):.1%} reduction)"
)

# Update the original DataFrames
customers = datasets["customers"]
items = datasets["items"]
payments = datasets["payments"]
reviews = datasets["reviews"]
orders_dataset = datasets["orders"]
products = datasets["products"]
sellers = datasets["sellers"]

customers    11.03 MB -> 10.65 MB
items        16.33 MB -> 14.72 MB
payments     8.11 MB -> 6.32 MB
reviews      18.50 MB -> 17.83 MB
orders       12.99 MB -> 12.99 MB
products     3.73 MB -> 2.60 MB
sellers      0.22 MB -> 0.21 MB

TOTAL: 70.92 MB -> 65.33 MB (7.9% reduction)


In [52]:
# 🔒 Fix Random Seed for Reproducibility (BERT Model)
import os, random
import numpy as np
import torch

SEED = 42

def set_seed(seed: int = SEED):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)
print(f"✔ Random seed fixed to {SEED} (Python, NumPy, PyTorch CPU & CUDA)")

✔ Random seed fixed to 42 (Python, NumPy, PyTorch CPU & CUDA)


In [ ]:
# Free the training-time BERT/CNN2D models, optimizers, and dataloaders before the
# appendix below (which loads its OWN fresh copies of BERT/CNN2D via ModelRegistry
# and a SHAP explainer). Without this, a full top-to-bottom "Run All" can hold two
# full BERT instances (~670MB each) plus optimizer state simultaneously and risk an
# out-of-memory crash on GPUs with limited VRAM -- found by actually running this
# notebook end-to-end, not assumed.
import gc

for _name in ["model_bert", "optimizer_bert", "scheduler_bert", "train_loader_bert",
              "val_loader_bert", "test_loader_bert", "model_cnn2d", "optimizer_cnn",
              "scheduler_cnn", "train_loader_cnn", "val_loader_cnn", "test_loader_cnn"]:
    if _name in globals():
        del globals()[_name]

gc.collect()
try:
    import torch as _torch
    if _torch.cuda.is_available():
        _torch.cuda.empty_cache()
        print(f"GPU memory freed. Currently allocated: {_torch.cuda.memory_allocated() / 1024**2:.1f} MB")
except ImportError:
    pass

import os
import sys
from pathlib import Path


def _find_repo_root(start: Path) -> Path:
    """This notebook lives outside the git repo (final code/Olist/), so locate
    Olist_Marketplace_Platform (the repo backing github.com/radwaelashry30-crypto/
    baseera-marketplace-analytics) to load its real results/*.json, models/, and
    backend source for the appendix below."""
    for base in [start, *start.parents]:
        candidate = base / "Olist_Marketplace_Platform"
        if (candidate / "backend" / "app").is_dir() and (candidate / "results").is_dir():
            return candidate
        if (base / "backend" / "app").is_dir() and (base / "results").is_dir():
            return base
    raise RuntimeError(
        "Could not locate the Olist_Marketplace_Platform repo near this notebook. "
        "Set REPO_ROOT manually to its path if it has been moved."
    )


REPO_ROOT = _find_repo_root(Path.cwd())
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT / "backend"))
print("Repo root for the appendix below:", REPO_ROOT)
